<div
     style="padding: 20px;
            color: white;
            font-size: 250%;
            text-align: center;
            display: fill;
            border-radius: 5px;
            background-color: #8be4a7ab;
            overflow: hidden;
            font-weight: 700;
            border: 5px solid #F28C28;"
     >
    GOTCHA! An Autoencoder Anomaly Detection of Fraud Supplimental Information
</div>

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Introduction to Autoencoders</b></p>
</div>

* An **autoencoder** is a neural network trained to reproduce its own input.

* It learns two functions:
    1. **Encoder:** compresses the input. We call the encoder $h$. The encoder will take the original x vector as input. We want the latent space $z = h(x)$. Ideally, $z >>> x$.
    2. **Decoder:** reconstructs the input. Wecall thedecoder $f$. The decoder will take an encoded latent space $f(z) =\tilde{x}$

* The goal: Have the smallest difference betwween $x$ and $\tilde{x}$.
* The compressed representation, $z$ is called the **latent space** or **bottleneck**. You could use $z$ as your data.
* The most common loss function used for autoencoders is MSE


<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Autoencoders in Keras</b></p>
</div>

In [ ]:
# imports
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

# autoencoder
AE = Sequential()
AE.add(Input(shape=(10,)))             ## [encoder] input layer with number of variables as neurons
AE.add(Dense(6, activation='relu'))    ## [encoder] hidden layer
AE.add(Dense(3, activation='relu'))    ## [encoder-decoder] latent space  ## [decoder] input layer with number of variables as neurons
AE.add(Dense(6, activation='relu'))    ## [decoder] hidden layer
AE.add(Dense(10, activation='linear')) ## [decoder] output layer

# loss function: MSE
AE.compile(optimizer='adam', loss='mse')

# summariize 
AE.summary ()

# fit
# AE.fit(X, X, epochs = 100, validation_split = 0.2, callbacks = [early_stops]) # (X_train, X_train)

# predict
# X_reconstructed = AE.predict(X)

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>What If You Want h & f Separately</b></p>
</div>

In [ ]:
## what if you want h (encoder NN) and f (decoder) separately

from tensorflow.keras.models import Model ## can combine different neural nets

# encoder (h)
encoder = Sequential(name = "Encoder")
encoder.add(Input(shape=(10,)))
encoder.add(Dense(6, activation='relu'))
encoder.add(Dense(3, activation='relu'))  ## latent space

# decoder (f)
decoder = Sequential(name = "Decoder")
decoder.add(Input(shape=(3,)))
decoder.add(Dense(6, activation='relu'))
decoder.add(Dense(10, activation='linear'))

# combine the two
AE1 = Model(inputs = encoder.inputs, outputs = decoder(encoder.outputs))

# compile
AE1.compile(optimizer='adam', loss='mse')

# summary
AE1.summary()

# fit
## AE1.fit(X, X, epochs = 100, validation_split = 0.2, callbacks = [early

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Architecture, Bottleneck, & Total Number of Parameters Calculation  </b></p>
</div>


* A simple autoencoder looks like:

**Input → Encoder → Latent Space → Decoder → Reconstruction**

* The input and output layers have the same number of variables because the goal is to reconstruct the original input.

* The bottleneck dimension is a three-dimensional latent space

* The output layer must have the same number of neurons as the predicting X

* The total number of parameters within an autoencoder can be calculated simply by using the code below:
```{python}
params_input_hidden = 6*4 + 4     
params_hidden_latent = 4*2 + 2
params_latent_hidden = 2*4 + 4
params_hidden_output = 4*6 + 6
total_params = params_input_hidden + params_hidden_latent + params_latent_hidden + params_hidden_output

print("Total parameters =", total_params)
```

* The output give the total number of parameters

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Autoencoder Loss Functions & Reconstruction Error</b></p>
</div>

* An autoencoder tries to reproduce the input rather than predict a separate target.
* A common loss for numeric inputs is **mean squared error (MSE)**:

$$
MSE = \frac{1}{n}\sum_{i=1}^{n}(x_i - \hat{x}_i)^2
$$

* Every observation has p-variables
* Reconstruction error is the MSE value by each row
* Calculation of loss function, given for this particular example it MSE can be derived as

```{python}
x_true = np.array([4.0, 2.0, 6.0])
x_hat = np.array([3.5, 2.2, 5.4])

obs_mse = np.mean((x_true - x_hat)**2) ## an mse by obs

print("Observation MSE =", obs_mse)
```

* Reconstruction error can be calculated by the MSE of each row
* If an observation has a much larger reconstruction error than the others, then this might suggest an anomaly has been detected.

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Tiny Encoder-Decoder Example</b></p>
</div>

```{python}
# One observation with 4 features
x = np.array([8.0, 3.0, 6.0, 1.0])

# Encoder: 4 input features -> 2 latent dimensions
W_enc = np.array([
    [0.20, 0.10],
    [0.40, 0.30],
    [0.10, 0.50],
    [0.30, 0.20]
])
b_enc = np.array([0.1, -0.2])

# Linear encoder transformation
z = x @ W_enc + b_enc

print("The two-dimensional latent space is:", z)

# Decoder: 2 latent dimensions -> 4 output features
W_dec = np.array([
    [0.50, 0.20, 0.30, 0.10],
    [0.10, 0.40, 0.20, 0.30]
])
b_dec = np.array([0.0, 0.1, -0.1, 0.2])

# Linear decoder transformation
x_hat = z @ W_dec + b_dec

```



<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Load Libraries & Data</b></p>
</div>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

# Set seed
np.random.seed(42)
tf.random.set_seed(42)

# Update the file name here if your CSV file is stored elsewhere.
file_path ='/content/insurance_fraud_dataset (1).csv'

# Import data
df = pd.read_csv(file_path)

# Examine data
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Check Fraud Distribution</b></p>
</div>

In [ ]:
print(df["FraudFound_P"].value_counts())
print("\nFraud rate:", round(df["FraudFound_P"].mean(), 4))


<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Define Features & Target</b></p>
</div>

In [ ]:
target_col = "FraudFound_P"

y = df[target_col].copy()
X = df.drop(columns=[target_col]).copy()

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Train / Test Split</b></p>
</div>

In [ ]:

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set shape:", X_train_full.shape)
print("Test set shape:", X_test.shape)

print("\nTraining fraud rate:", round(y_train_full.mean(), 4))
print("Test fraud rate:", round(y_test.mean(), 4))

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Train Only on Normal Claims</b></p>
</div>

* The autoencoder is trained only on non-fraud claims so it learns what typical claims look like/

In [ ]:
X_train_normal = X_train_full[y_train_full == 0].copy()

print("Non-fraud training rows:", X_train_normal.shape[0])

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Scale the Data</b></p>
</div>

* Scaling matter because reconsturction error is based on all features together
* Without scaling, variables with larger ranges would dominate loss function

In [ ]:
scaler = StandardScaler()

X_train_normal_scaled = scaler.fit_transform(X_train_normal)
X_train_full_scaled = scaler.transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

print("Scaled normal training shape:", X_train_normal_scaled.shape)
print("Scaled full training shape:", X_train_full_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Build Main Autoencoder</b></p>
</div>

In [ ]:
input_dim = X_train_normal_scaled.shape[1]

input_layer = keras.Input(shape=(input_dim,))
encoded = layers.Dense(64, activation="relu")(input_layer)
bottleneck = layers.Dense(16, activation="relu")(encoded)
decoded = layers.Dense(64, activation="relu")(bottleneck)
output_layer = layers.Dense(input_dim, activation="linear")(decoded)

autoencoder = keras.Model(inputs=input_layer, 
                          outputs=output_layer)

autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

autoencoder.summary()

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Train Main Autoencoder</b></p>
</div>

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = autoencoder.fit(
    X_train_normal_scaled,
    X_train_normal_scaled,
    epochs=100,
    batch_size=64,
    validation_split=0.20,
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Plot Training History</b></p>
</div>

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Main Autoencoder Training History")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.show()


<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Reconstruction Error Function</b></p>
</div>

In [ ]:
def get_reconstruction_error(model, X_scaled):
    reconstructed = model.predict(X_scaled, verbose=0)
    mse = np.mean(np.square(X_scaled - reconstructed), axis=1)
    return mse


<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Calculation of Reconstruction Error</b></p>
</div>

In [ ]:
train_normal_error = get_reconstruction_error(autoencoder, X_train_normal_scaled)
train_full_error = get_reconstruction_error(autoencoder, X_train_full_scaled)
test_error = get_reconstruction_error(autoencoder, X_test_scaled)

print("Train normal reconstruction error summary:")
print(pd.Series(train_normal_error).describe())

print("\nTest reconstruction error summary:")
print(pd.Series(test_error).describe())

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Create Fraud Risk Scores</b></p>
</div>

In [ ]:
train_results = X_train_full.copy()
train_results["FraudFound_P"] = y_train_full.values
train_results["risk_score"] = train_full_error

test_results = X_test.copy()
test_results["FraudFound_P"] = y_test.values
test_results["risk_score"] = test_error

# Keep original test index for later joins and plots
test_results["original_index"] = test_results.index

# Rank from highest risk to lowest
test_results_ranked = test_results.sort_values(
    "risk_score",
    ascending=False
).reset_index(drop=True)

display(test_results_ranked.head(10))

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Plot Reconstruction Error Distribution</b></p>
</div>

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(test_results["risk_score"], bins=50)
plt.title("Distribution of Reconstruction Error (Test Set)")
plt.xlabel("Reconstruction Error")
plt.ylabel("Count")
plt.show()

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Compare Fraud vs Non-Fraud Error</b></p>
</div>

In [ ]:
fraud_scores = test_results.loc[test_results["FraudFound_P"] == 1, "risk_score"]
nonfraud_scores = test_results.loc[test_results["FraudFound_P"] == 0, "risk_score"]

plt.figure(figsize=(8, 5))
plt.hist(nonfraud_scores, bins=50, alpha=0.7, label="Non-Fraud")
plt.hist(fraud_scores, bins=50, alpha=0.7, label="Fraud")
plt.title("Reconstruction Error: Fraud vs Non-Fraud")
plt.xlabel("Reconstruction Error")
plt.ylabel("Count")
plt.legend()
plt.show()

print("Average reconstruction error (Non-Fraud):", round(nonfraud_scores.mean(), 4))
print("Average reconstruction error (Fraud):", round(fraud_scores.mean(), 4))

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Top-Risk Investigation Strategy</b></p>
</div>

* The project guide suggests selecting a target group such as the 1000 claims most likely to be fraudulent

In [ ]:
top_n = min(1000, len(test_results_ranked))
top_risk_claims = test_results_ranked.head(top_n).copy()

print("Selected top-risk group size:", top_n)
print("Actual frauds captured in top-risk group:", int(top_risk_claims["FraudFound_P"].sum()))
print("Fraud rate within top-risk group:", round(top_risk_claims["FraudFound_P"].mean(), 4))

display(top_risk_claims.head(10))

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Evalaute Business Usefulness</b></p>
</div>

In [ ]:
frauds_captured = int(top_risk_claims["FraudFound_P"].sum())
total_actual_frauds_in_test = int(y_test.sum())

precision_at_top_n = frauds_captured / top_n
recall_at_top_n = frauds_captured / total_actual_frauds_in_test

overall_test_fraud_rate = y_test.mean()
expected_frauds_random = top_n * overall_test_fraud_rate
lift = precision_at_top_n / overall_test_fraud_rate

print("========== BUSINESS EVALUATION ==========")
print("Top N selected:", top_n)
print("Actual frauds captured:", frauds_captured)
print("Total actual frauds in test set:", total_actual_frauds_in_test)
print("Precision in selected group:", round(precision_at_top_n, 4))
print("Recall in selected group:", round(recall_at_top_n, 4))
print("Overall fraud rate in test set:", round(overall_test_fraud_rate, 4))
print("Expected frauds under random selection:", round(expected_frauds_random, 2))
print("Lift over random baseline:", round(lift, 2), "x")

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Threhold-Based View</b></p>
</div>

* The project is mainly about ranking, but this optional section give a threshold-based anomaly flag for added interpretation

In [ ]:
threshold = np.percentile(train_normal_error, 95)
print("Threshold (95th percentile of normal training error):", round(threshold, 4))

test_results["predicted_anomaly"] = (test_results["risk_score"] > threshold).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(test_results["FraudFound_P"], test_results["predicted_anomaly"]))

print("\nClassification Report:")
print(classification_report(test_results["FraudFound_P"], test_results["predicted_anomaly"], digits=4))

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Summary Table</b></p>
</div>

In [ ]:
summary_table = pd.DataFrame({
    "Metric": [
        "Top-N claims selected",
        "Frauds captured in Top-N",
        "Precision in Top-N",
        "Recall in Top-N",
        "Overall fraud rate in test set",
        "Expected frauds under random selection",
        "Lift over random selection"
    ],
    "Value": [
        top_n,
        frauds_captured,
        round(precision_at_top_n, 4),
        round(recall_at_top_n, 4),
        round(overall_test_fraud_rate, 4),
        round(expected_frauds_random, 2),
        round(lift, 2)
    ]
})

display(summary_table)

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Build 2D Autoenconder</b></p>
</div>

In [ ]:
latent_dim = 2

input_layer_2d = keras.Input(shape=(input_dim,))
encoded_2d = layers.Dense(32, activation="relu")(input_layer_2d)
latent_2d = layers.Dense(latent_dim, activation="linear", name="latent_space")(encoded_2d)
decoded_2d = layers.Dense(32, activation="relu")(latent_2d)
output_layer_2d = layers.Dense(input_dim, activation="linear")(decoded_2d)

autoencoder_2d = keras.Model(inputs=input_layer_2d, outputs=output_layer_2d)
encoder_2d = keras.Model(inputs=input_layer_2d, outputs=latent_2d)

autoencoder_2d.compile(
    optimizer="adam",
    loss="mse"
)

autoencoder_2d.summary()

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Train 2D Autoencoder</b></p>
</div>

In [ ]:
history_2d = autoencoder_2d.fit(
    X_train_normal_scaled,
    X_train_normal_scaled,
    epochs=100,
    batch_size=64,
    validation_split=0.20,
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Extract 2D Latent Representation</b></p>
</div>

In [ ]:
latent_test = encoder_2d.predict(X_test_scaled, verbose=0)

latent_df = pd.DataFrame(latent_test, columns=["Latent_1", "Latent_2"], index=X_test.index)
latent_df["FraudFound_P"] = y_test
latent_df["risk_score"] = test_results["risk_score"]
latent_df["Top_Risk_Flag"] = 0

top_risk_original_idx = top_risk_claims["original_index"].values
latent_df.loc[top_risk_original_idx, "Top_Risk_Flag"] = 1

display(latent_df.head())

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Plot 2D Latent Space: Fraud vs Non-Fraud</b></p>
</div>

In [ ]:
plt.figure(figsize=(9, 7))

nonfraud_points = latent_df[latent_df["FraudFound_P"] == 0]
fraud_points = latent_df[latent_df["FraudFound_P"] == 1]

plt.scatter(
    nonfraud_points["Latent_1"],
    nonfraud_points["Latent_2"],
    alpha=0.4,
    s=25,
    label="Non-Fraud"
)

plt.scatter(
    fraud_points["Latent_1"],
    fraud_points["Latent_2"],
    alpha=0.9,
    s=60,
    marker="x",
    label="Fraud"
)

plt.title("2D Latent Space of Claims")
plt.xlabel("Latent Dimension 1")
plt.ylabel("Latent Dimension 2")
plt.legend()
plt.show()

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Plot 2D Latent Space: Top-Risk Claims</b></p>
</div>

In [ ]:
plt.figure(figsize=(9, 7))

base_points = latent_df[latent_df["Top_Risk_Flag"] == 0]
top_points = latent_df[latent_df["Top_Risk_Flag"] == 1]

plt.scatter(
    base_points["Latent_1"],
    base_points["Latent_2"],
    alpha=0.3,
    s=20,
    label="Other Claims"
)

plt.scatter(
    top_points["Latent_1"],
    top_points["Latent_2"],
    alpha=0.9,
    s=50,
    marker="o",
    label="Top-Risk Claims"
)

plt.title("2D Latent Space with Top-Risk Claims Highlighted")
plt.xlabel("Latent Dimension 1")
plt.ylabel("Latent Dimension 2")
plt.legend()
plt.show()

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Plot 2D Latent Space: Colored by Risk Score</b></p>
</div>

In [ ]:
plt.figure(figsize=(9, 7))
scatter = plt.scatter(
    latent_df["Latent_1"],
    latent_df["Latent_2"],
    c=latent_df["risk_score"],
    alpha=0.7,
    s=30
)
plt.colorbar(scatter, label="Reconstruction Error / Risk Score")
plt.title("2D Latent Space Colored by Risk Score")
plt.xlabel("Latent Dimension 1")
plt.ylabel("Latent Dimension 2")
plt.show()

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Top-Risk Review Table</b></p>
</div>

In [ ]:
review_columns = list(X.columns[:10])
review_table = top_risk_claims[review_columns + ["FraudFound_P", "risk_score"]].copy()

print("Top-risk review table:")
display(review_table.head(20))


<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Percentile Check</b></p>
</div>

* This is useful for showing how fraud concentration changes at stricter cutoffs

In [ ]:
for p in [90, 95, 97, 99]:
    cutoff = np.percentile(test_results["risk_score"], p)
    flagged = test_results[test_results["risk_score"] >= cutoff]
    print(
        f"Percentile {p}:",
        "Claims =", len(flagged),
        "| Fraud Rate =", round(flagged["FraudFound_P"].mean(), 4)
    )

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Stability Check</b></p>
</div>

In [ ]:
top_500_ids = set(test_results.sort_values("risk_score", ascending=False).head(500).index)
top_1000_ids = set(test_results.sort_values("risk_score", ascending=False).head(1000).index)

overlap_ratio = len(top_500_ids.intersection(top_1000_ids)) / len(top_500_ids)
print("Overlap of Top 500 within Top 1000:", round(overlap_ratio, 4))

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Final Results Summary</b></p>
</div>

In [ ]:
print("===================================================")
print("FINAL RESULTS SUMMARY")
print("===================================================")
print(f"Test Set Size: {len(test_results)}")
print(f"Fraud Rate in Test Set: {overall_test_fraud_rate:.4f}")
print(f"Top-{top_n} Risk Group Fraud Rate: {precision_at_top_n:.4f}")
print(f"Frauds Captured in Top-{top_n}: {frauds_captured}")
print(f"Recall in Top-{top_n}: {recall_at_top_n:.4f}")
print(f"Expected Frauds Under Random Selection: {expected_frauds_random:.2f}")
print(f"Lift Over Random Selection: {lift:.2f}x")
print("===================================================")

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Save Results</b></p>
</div>

In [ ]:
test_results_ranked.to_csv("ranked_test_claims_by_risk.csv", index=False)
top_risk_claims.to_csv("top_risk_claims_selected.csv", index=False)

print("Saved:")
print("ranked_test_claims_by_risk.csv")
print("top_risk_claims_selected.csv")